# 04 · Empaquetamiento

El notebook 03 dejó un modelo que vive dentro de un notebook. Para que la API lo pueda
servir tiene que ser un **paquete instalable**: `pip install edad_cerebral` y ya.

Eso es lo que hace este notebook, y es el patrón del Taller 5 del curso.

| | |
|---|---|
| Entra | las dos tablas de `02` y la partición de `01` |
| Sale | `edad_cerebral-0.1.0-py3-none-any.whl` |
| Tarda | ~2 min |

Tres cosas que este notebook **verifica antes de dar nada por bueno**:

1. Que el artefacto reproduce **7,48 / 10,43 / 9,15**. Si no, aborta.
2. Que extraer las características desde el EDF **con el código del paquete** da
   exactamente las mismas columnas que produjo el notebook 02.
3. Que el wheel, instalado en un entorno limpio, predice.


In [1]:
import json, subprocess, sys
from pathlib import Path

import numpy as np, pandas as pd

# El paquete vive en el repositorio de la aplicación, no aquí.
PAQUETE = next(p / 'SomnoAI' / 'package-src'
               for p in Path.cwd().parents if (p / 'SomnoAI' / 'package-src').is_dir())
DATOS_EDF = next(p / 'Microproyecto' / 'Data' / 'sleep-cassette'
                 for p in Path.cwd().parents if (p / 'Microproyecto' / 'Data').is_dir())
sys.path.insert(0, str(PAQUETE))

print('paquete :', PAQUETE)
print('EDF     :', DATOS_EDF)


paquete : /Users/sebastian/Library/Mobile Documents/com~apple~CloudDocs/Maestria Andes/Cuarto semetre/Proyecto - Desarrollo de Soluciones/Microproyectos/SomnoAI/package-src
EDF     : /Users/sebastian/Library/Mobile Documents/com~apple~CloudDocs/Maestria Andes/Cuarto semetre/Proyecto - Desarrollo de Soluciones/Microproyectos/Microproyecto/Data/sleep-cassette


## De dónde sale cada archivo

El paquete no es código nuevo: es el de los notebooks 02 y 03, movido a módulos.

| Del notebook | Celda | Va a |
|---|---|---|
| `02_caracteristicas` | `espectro`, `doce_numeros`, `arquitectura` | `processing/features.py` |
| `02_caracteristicas` | `cargar_noche`, `recortar`, `espectros_por_fase` | `processing/edf.py` |
| `03_modelo` | `RidgeBAI` | `ridge_bai.py` |
| `03_modelo` | `pipeline()` | `pipeline.py` |
| `03_modelo` | `por_sujeto`, la tabla de modelado, el ajuste | `train_pipeline.py` |
| `03_modelo` | `ESPECTRALES`, `ARQUITECTURA`, `alpha`, `lambda` | `config.yml` |

Lo único que **no** se lleva son las curvas por fase: el modelo usa 32 columnas
(12 `mix_*` por canal + 8 de arquitectura), no las 97 del CSV. Calcular las otras 65
en producción sería trabajo tirado.


In [2]:
# Las rutas de iCloud llevan espacios: nada de partir la salida de `find`.
for ruta in sorted(p for p in (PAQUETE / 'edad_cerebral').rglob('*')
                   if p.suffix in ('.py', '.yml', '.pkl') or p.name == 'VERSION'):
    if '__pycache__' in ruta.parts:
        continue
    print(f'  {ruta.relative_to(PAQUETE)}')


  edad_cerebral/VERSION
  edad_cerebral/__init__.py
  edad_cerebral/config/__init__.py
  edad_cerebral/config/core.py
  edad_cerebral/config.yml
  edad_cerebral/datasets/__init__.py
  edad_cerebral/norms.py
  edad_cerebral/pipeline.py
  edad_cerebral/predict.py
  edad_cerebral/processing/__init__.py
  edad_cerebral/processing/data_manager.py
  edad_cerebral/processing/edf.py
  edad_cerebral/processing/features.py
  edad_cerebral/processing/validation.py
  edad_cerebral/ridge_bai.py
  edad_cerebral/train_pipeline.py
  edad_cerebral/trained/__init__.py
  edad_cerebral/trained/edad_cerebral_v0.3.0.pkl


## El artefacto

`train_pipeline.py` reconstruye el experimento C: promedia las noches de cada sujeto,
lee la partición, y ajusta `Pipeline(imputar → escalar → RidgeBAI)` sobre los **62
sujetos de desarrollo**.

No se entrena con los 78. El test se deja intacto para que el 10,43 y el 9,15 del
reporte sigan siendo los del modelo que se despliega, y no de uno parecido.

**Si las métricas no cuadran con las del reporte, no guarda nada y lanza.** Es la misma
disciplina de `01_particion`: antes que un artefacto silenciosamente distinto, un fallo
ruidoso.


In [3]:
from edad_cerebral.train_pipeline import run_training

r = run_training()
print(f"{r['n_desarrollo']} sujetos de desarrollo · {r['n_columnas']} columnas\n")
for k, v in r['metricas'].items():
    print(f'  {k:16s} {v:5.2f}')
print(f"\nartefacto: {Path(r['ruta']).relative_to(PAQUETE)}")


62 sujetos de desarrollo · 32 columnas

  mae_train         7.48
  mae_validacion   10.43
  mae_test          9.15

artefacto: edad_cerebral/trained/edad_cerebral_v0.3.0.pkl


## La prueba que importa

Empaquetar sale mal de una forma muy concreta: el código se reescribe, sigue pareciendo
correcto, y produce números **ligeramente** distintos. Como el `StandardScaler` está
ajustado sobre los números del notebook 02, cualquier desplazamiento corre la predicción
**sin lanzar ningún error**. Sale una edad plausible y equivocada.

Así que se comprueba de frente: extraer una noche desde el EDF con el código del paquete
y comparar contra la fila que el notebook 02 escribió en el CSV.


In [4]:
from edad_cerebral.config.core import DATASET_DIR, config
from edad_cerebral.predict import extraer_caracteristicas

COD = 'SC4001E0'
hyp = sorted(DATOS_EDF.glob(f'{COD[:6]}*-Hypnogram.edf'))[0]
obtenidas = extraer_caracteristicas(str(DATOS_EDF / f'{COD}-PSG.edf'), str(hyp))

fpz = pd.read_csv(DATASET_DIR / config.app_config.csv_fpz)
pz  = pd.read_csv(DATASET_DIR / config.app_config.csv_pz)
f_fpz = fpz[fpz.registro == COD].iloc[0]
f_pz  = pz[pz.registro == COD].iloc[0]

filas = []
for e in config.modelo.espectrales:
    filas.append((f'mix_{e}_FpzCz', obtenidas[f'mix_{e}_FpzCz'], f_fpz[f'mix_{e}']))
    filas.append((f'mix_{e}_PzOz',  obtenidas[f'mix_{e}_PzOz'],  f_pz[f'mix_{e}']))
for a in config.modelo.arquitectura:
    filas.append((a, obtenidas[a], f_fpz[a]))

comp = pd.DataFrame(filas, columns=['columna', 'paquete', 'notebook_02'])
comp['dif'] = (comp.paquete - comp.notebook_02).abs()

print(f'{len(comp)} columnas comparadas · diferencia máxima = {comp.dif.max():.2e}')
assert comp.dif.max() < 1e-9, 'el paquete NO reproduce el notebook 02'
print('el paquete reproduce el notebook 02\n')
comp.sort_values('dif', ascending=False).head(8).round(6)


32 columnas comparadas · diferencia máxima = 4.44e-16
el paquete reproduce el notebook 02



,columna,paquete,notebook_02,dif
0,mix_potencia_total_log_FpzCz,3.261506,3.261506,0.0
8,mix_abs_sigma_FpzCz,1.135720,1.135720,0.0
25,eficiencia,0.905687,0.905687,0.0
16,mix_rel_alpha_FpzCz,0.014344,0.014344,0.0
20,mix_rel_beta_FpzCz,0.004448,0.004448,0.0
21,mix_rel_beta_PzOz,0.004592,0.004592,0.0
19,mix_rel_sigma_PzOz,0.006598,0.006598,0.0
17,mix_rel_alpha_PzOz,0.029816,0.029816,0.0


## Construir el wheel

`python -m build` deja en `dist/` un `.tar.gz` (el código fuente) y un `.whl` (el
instalable). El `.whl` es el que consume la API.

Dentro viajan el artefacto `.pkl`, el `config.yml` y las tablas de `datasets/`: sin ellos
el paquete instalado no podría ni predecir ni reentrenarse.


In [5]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'build'], check=True)
res = subprocess.run([sys.executable, '-m', 'build'], cwd=PAQUETE,
                     capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1])

import zipfile
whl = sorted((PAQUETE / 'dist').glob('*.whl'))[-1]
print(f'\n{whl.name}  ({whl.stat().st_size/1024:.0f} KB)\n')
for n in sorted(zipfile.ZipFile(whl).namelist()):
    if not n.startswith('edad_cerebral-0'):
        print('  ', n)


Successfully built edad_cerebral-0.3.0.tar.gz and edad_cerebral-0.3.0-py3-none-any.whl

edad_cerebral-0.3.0-py3-none-any.whl  (305 KB)

   edad_cerebral/VERSION
   edad_cerebral/__init__.py
   edad_cerebral/config.yml
   edad_cerebral/config/__init__.py
   edad_cerebral/config/core.py
   edad_cerebral/datasets/__init__.py
   edad_cerebral/datasets/caracteristicas_noche_FpzCz.csv
   edad_cerebral/datasets/caracteristicas_noche_PzOz.csv
   edad_cerebral/datasets/espectros_nrem_FpzCz.csv
   edad_cerebral/datasets/subject_split_seed42.json
   edad_cerebral/norms.py
   edad_cerebral/pipeline.py
   edad_cerebral/predict.py
   edad_cerebral/processing/__init__.py
   edad_cerebral/processing/data_manager.py
   edad_cerebral/processing/edf.py
   edad_cerebral/processing/features.py
   edad_cerebral/processing/validation.py
   edad_cerebral/ridge_bai.py
   edad_cerebral/train_pipeline.py
   edad_cerebral/trained/__init__.py
   edad_cerebral/trained/edad_cerebral_v0.3.0.pkl


## Que funcione instalado, no solo aquí

El fallo clásico al empaquetar un estimador propio: el `Pipeline` serializado guarda la
**ruta de importación** de la clase (`edad_cerebral.ridge_bai.RidgeBAI`). Si esa clase
solo existiera en un notebook, el `.pkl` no se podría abrir en ningún otro sitio.

Se comprueba instalando el wheel en un entorno virgen y prediciendo desde allí.


In [6]:
import os, tempfile

tmp = Path(tempfile.mkdtemp())
entorno = tmp / 'venv'
subprocess.run([sys.executable, '-m', 'venv', str(entorno)], check=True)
py = entorno / 'bin' / 'python'
subprocess.run([str(py), '-m', 'pip', 'install', '-q', str(whl)], check=True)

# Limpio de verdad: sin esto el subproceso hereda MPLBACKEND del kernel de
# Jupyter, que apunta a matplotlib_inline. Ese paquete existe aqui pero no en el
# entorno nuevo, asi que mne (que arrastra matplotlib) reventaria al importarse
# y pareceria un fallo del empaquetamiento sin serlo.
limpio = {k: v for k, v in os.environ.items()
          if k not in ('MPLBACKEND', 'PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP')}

# El guion va a un archivo y no a `-c`: las rutas llevan espacios y comillas.
guion = tmp / 'probar.py'
guion.write_text(f'''
from edad_cerebral import __version__
from edad_cerebral.predict import predecir_desde_edf

r = predecir_desde_edf({str(DATOS_EDF / f'{COD}-PSG.edf')!r}, {str(hyp)!r})
print('paquete       ', __version__)
print('modelo        ', r['version'])
print('edad cerebral ', round(r['edad_cerebral'], 1), 'anios')
print('error tipico  ', r['error_tipico'], f"({{r['nivel_intervalo']:.0%}})")
''')

res = subprocess.run([str(py), str(guion)], capture_output=True, text=True, env=limpio)
print(res.stdout)
if res.returncode:                      # sin esto, un fallo pasaria desapercibido
    print(res.stderr)
    raise RuntimeError('el paquete instalado no predice')



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /private/var/folders/83/lhxwld796l7cx3lyt07ml50m0000gn/T/tmpgiywazvq/venv/bin/python -m pip install --upgrade pip


paquete        0.3.0
modelo         edad-cerebral-0.3.0
edad cerebral  24.8 anios
error tipico   10.43 (90%)



## Cómo lo consume la API

El wheel se copia a `SomnoAI/model-pkg/` y `backend/requirements.txt` lo declara en su
**primera línea**, igual que en el Taller 6:

```
./model-pkg/edad_cerebral-0.1.0-py3-none-any.whl
uvicorn==...
fastapi==...
```

A partir de ahí la API solo hace:

```python
from edad_cerebral.predict import predecir_desde_edf
```

El día que se reentrene: se sube la `VERSION`, se reconstruye el wheel con este notebook,
se cambia esa línea del `requirements.txt` y se reconstruye la imagen. Nada de código.

---

### Una nota sobre las versiones

`requirements/requirements.txt` del paquete fija las versiones del **entorno de servicio**,
no las del portátil. Las tablas de `datasets/` se calcularon con un `scipy` concreto y el
escalador del modelo está ajustado sobre esos números: extraer en producción con otra
versión movería las características sin dar ningún error.

Por eso la prueba de ida y vuelta de arriba hay que correrla también **dentro del
contenedor**, no solo aquí.
